# Machine Learning Cycle — MVTec AD Anomaly Detection

**Objective:** Demonstrate the end-to-end ML process for manufacturing defect detection using the MVTec AD dataset (image data, non-tabular).

This notebook covers:
1. Data acquisition
2. Data preprocessing
3. Model creation (Convolutional Autoencoder, trained on normal images only)
4. Model testing & evaluation (≥4 metrics: accuracy, loss, F1, precision, recall)
5. Visualizations (≥3 feature insights)
6. Single-image prediction demo

## 1. Imports & Configuration

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
from src.preprocessing import acquire_and_prepare, load_processed, IMG_SIZE
from src.model import ConvAutoencoder, train_model, anomaly_score, select_threshold, predict, load_model
from src.prediction import load_threshold, save_threshold, predict_single

CATEGORY = "bottle"   # change to any MVTec AD category: hazelnut, screw, etc.
LIMIT = 300            # set None to use full dataset

print(f"Category: {CATEGORY}")
print(f"Using device: {'cuda' if __import__('torch').cuda.is_available() else 'cpu'}")

## 2. Data Acquisition

Load raw MVTec AD images for one category. Training uses **normal (defect-free) images only** — this is unsupervised anomaly detection: the model learns what "normal" looks like.

In [ ]:
# Point RAW_DATA_ROOT to your extracted MVTec AD folder in src/preprocessing.py first,
# then run this cell to acquire + preprocess.

X_train, X_test, y_test, test_paths = acquire_and_prepare(CATEGORY, limit=LIMIT)

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Test labels  normal={(y_test==0).sum()}, anomalous={(y_test==1).sum()}")

## 3. Data Preprocessing

- Resize all images to 224×224
- Normalize pixel values to [0, 1]
- Convert to float32 numpy arrays
- Train set: normal images only (for unsupervised AD)
- Test set: mixed normal + anomalous (for evaluation)

In [ ]:
# Show a few training (normal) images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i])
    ax.axis('off')
    ax.set_title(f'Normal #{i}')
plt.suptitle('Training images (normal / defect-free only)')
plt.show()

# Show a few test images with labels
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i])
    ax.axis('off')
    ax.set_title(f'Label: {"Normal" if y_test[i]==0 else "Anomalous"}')
plt.suptitle('Test images (mixed normal + anomalous)')
plt.show()

## 4. Model Creation — Convolutional Autoencoder

The autoencoder is trained **only on normal images**. At inference, reconstruction error (MSE) is the anomaly score: higher error → more likely anomalous.

Optimization techniques used:
- Adam optimizer
- Early stopping (monitor validation loss)
- Sigmoid output activation (bounded reconstruction)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3

X_fit, X_validation = train_test_split(X_train, test_size=0.2, random_state=42)
model, history = train_model(
    X_fit,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    validation_data=X_validation,
)

# Plot training loss
plt.figure(figsize=(6, 4))
plt.plot(history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training Loss (normal images only)')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Select Anomaly Threshold

Threshold = 90th percentile of reconstruction errors on a held-out subset of normal training images, favoring defect recall.

In [ ]:
scores_normal, recon_train = anomaly_score(model, X_validation)
threshold = select_threshold(scores_normal, percentile=90)
save_threshold(threshold)

print(f"Selected threshold (p90): {threshold:.6f}")

## 6. Model Testing & Evaluation

Compute **≥4 evaluation metrics**: accuracy, loss (MSE), F1-score, precision, recall (+ ROC-AUC).

In [ ]:
labels, scores = predict(model, X_test, threshold)

accuracy  = accuracy_score(y_test, labels)
precision = precision_score(y_test, labels, zero_division=0)
recall    = recall_score(y_test, labels, zero_division=0)
f1        = f1_score(y_test, labels, zero_division=0)
roc_auc   = roc_auc_score(y_test, scores)

print("=== Evaluation Metrics ===")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, labels))

## 7. Visualizations (≥3 Feature Insights)

Insight 1 — **Reconstruction quality**: normal images reconstruct well; anomalous ones do not.
Insight 2 — **Error heatmap**: pixel-level defect localization.
Insight 3 — **Score distribution**: separation between normal and anomalous scores with the threshold line.

In [ ]:
_, recon_test = anomaly_score(model, X_test)

# Pick one normal and one anomalous test image
idx_normal = int(np.where(y_test == 0)[0][0])
idx_anom   = int(np.where(y_test == 1)[0][0]) if (y_test == 1).any() else idx_normal

for title, idx in [("Normal", idx_normal), ("Anomalous", idx_anom)]:
    orig = X_test[idx]
    rec  = np.transpose(recon_test[idx], (1, 2, 0))   # (C,H,W)->(H,W,C)
    err  = ((orig - rec) ** 2).mean(axis=2)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig); axes[0].set_title(f'Original ({title})'); axes[0].axis('off')
    axes[1].imshow(rec);  axes[1].set_title('Reconstruction'); axes[1].axis('off')
    im = axes[2].imshow(err, cmap='hot'); axes[2].set_title('Error Heatmap'); axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    plt.suptitle(f'{title} image — reconstruction & pixel error')
    plt.show()

# Score distribution
plt.figure(figsize=(7, 4))
plt.hist(scores[y_test == 0], bins=30, alpha=0.6, label='Normal', color='green')
plt.hist(scores[y_test == 1], bins=30, alpha=0.6, label='Anomalous', color='red')
plt.axvline(threshold, color='black', linestyle='--', label=f'Threshold = {threshold:.4f}')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.title('Anomaly Score Distribution with Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Single-Image Prediction Demo

Run the prediction pipeline on one test image (mirrors what the API / UI does).

In [ ]:
# Predict on the first anomalous test image
demo_idx = int(np.where(y_test == 1)[0][0]) if (y_test == 1).any() else 0
demo_path = test_paths[demo_idx]

result = predict_single(demo_path, include_images=True)

print(f"File: {demo_path}")
print(f"True label: {'Anomalous' if y_test[demo_idx]==1 else 'Normal'}")
print(f"Prediction: {'ANOMALOUS' if result['anomaly'] else 'NORMAL'}")
print(f"Score: {result['score']:.6f} | Threshold: {result['threshold']:.6f}")
print(f"Confidence: {result['confidence']:.2%}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(Image.open(demo_path).convert('RGB').resize((224, 224)))
axes[0].set_title('Input image')
axes[1].imshow(result['reconstruction'])
axes[1].set_title('Model reconstruction')
heatmap = axes[2].imshow(result['error_map'], cmap='hot')
axes[2].set_title('Pixel error heatmap')
for ax in axes:
    ax.axis('off')
fig.colorbar(heatmap, ax=axes[2], fraction=0.046)
plt.suptitle(f"Prediction: {'ANOMALOUS' if result['anomaly'] else 'NORMAL'} (score={result['score']:.4f})")
plt.tight_layout()
plt.show()

## Conclusion

The Convolutional Autoencoder trained exclusively on normal MVTec AD images successfully separates normal from anomalous parts using reconstruction error, with a data-driven threshold. The pipeline (preprocess → train → evaluate → predict → retrain) is fully implemented in `src/` and exposed via Flask API + Streamlit UI, ready for Docker deployment and Locust load testing as required by the assignment.